# 05 Final Load Preparation
Aggregating metrics and final dataset preparation to optimize dashboard performance in Tableau.

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('data/processed/cleaned_dataset.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Ensure calculated columns exist
if 'gross_profit_margin_pct' not in df.columns:
    df['gross_profit_margin_pct'] = (df['profit'] / df['sales']) * 100
if 'shipping_days' not in df.columns:
    df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days


## 1. Creating the Final Tableau Flat Table
The existing dataset is robust. We will save out a designated `tableau_ready.csv` that contains refined business KPIs.

In [ ]:
final_columns = [
    'order_id', 'order_date', 'order_year', 'order_month',
    'customer_name', 'customer_segment', 'city', 'state', 'region',
    'product_category', 'product_sub_category', 'product_name',
    'sales', 'profit', 'gross_profit_margin_pct', 'order_quantity',
    'discount', 'shipping_cost', 'shipping_days', 'ship_mode', 'order_priority'
]
tableau_df = df[[c for c in final_columns if c in df.columns]]

os.makedirs('data/processed', exist_ok=True)
tableau_df.to_csv('data/processed/tableau_ready_dataset.csv', index=False)
print(f'Exported {tableau_df.shape[0]} rows to tableau_ready_dataset.csv')

## 2. Aggregate Summaries for Auxiliary Dashboard Data
Providing grouped datasets if needed for particular mapping or trend optimizations.

In [ ]:
monthly_kpi = tableau_df.groupby(['order_year', 'order_month']).agg({
    'sales': 'sum',
    'profit': 'sum',
    'order_quantity': 'sum'
}).reset_index()

monthly_kpi['monthly_revenue_growth_pct'] = monthly_kpi['sales'].pct_change() * 100
monthly_kpi.to_csv('data/processed/monthly_kpis.csv', index=False)
print('Exported Monthly KPIs')